# SMAQ Benchmark — Colab Runner

Runs `benchmark.py` for:
- **Qwen/Qwen2.5-0.5B** (dense, 24 layers)
- **Qwen/Qwen3.5-0.8B** (hybrid, 6 full-attention layers)

Based on [TurboQuant](https://github.com/0xSero/turboquant) architecture.

In [1]:
# 1. Install dependencies (including latest transformers for Qwen 3.5 support)
!pip install -q torch datasets tabulate
!pip install -q git+https://github.com/huggingface/transformers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [10]:
"""
SMAQ Benchmark: Qwen2.5-0.5B and Qwen3.5-0.8B
================================================
Supports two models:
  - Qwen/Qwen2.5-0.5B (dense transformer, 24 layers, all full-attention)
  - Qwen/Qwen3.5-0.8B (hybrid: 24 layers, only 6 full_attention at layers 3,7,11,15,19,23)

Based on [TurboQuant](https://github.com/0xSero/turboquant) architecture.
"""

import os
import torch
import time
import gc
import sys

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tabulate import tabulate

device = "cuda" if torch.cuda.is_available() else "cpu"

MODELS = {
    "Qwen2.5-0.5B": {
        "id": "Qwen/Qwen2.5-0.5B",
        "dtype": torch.float16 if device == "cuda" else torch.float32,
        "all_layers_are_full_attn": True,
        "block_dim": 8,
    },
    "Qwen3.5-0.8B": {
        "id": "Qwen/Qwen3.5-0.8B",
        "dtype": torch.float16 if device == "cuda" else torch.float32,
        "all_layers_are_full_attn": False,
        "full_attention_layers": [3, 7, 11, 15, 19, 23],
        "block_dim": 8,
    },
}

NUM_SAMPLES = 8
SEQ_LEN = 128
N_CENTROIDS = 256
KMEANS_ITERS = 20
SEED = 42

def kmeans(data, n_centroids, n_iters, seed):
    N, D = data.shape
    rng = torch.Generator().manual_seed(seed)
    if N <= n_centroids:
        res = torch.zeros((n_centroids, D), device=data.device, dtype=data.dtype)
        res[:N] = data
        return res

    indices = [torch.randint(N, (1,), generator=rng).item()]
    for _ in range(n_centroids - 1):
        dists = torch.cdist(data, data[indices]).min(dim=1).values
        probs = dists**2
        p_sum = probs.sum()
        probs = probs / p_sum if p_sum > 0 else torch.ones(N, device=data.device) / N
        indices.append(torch.multinomial(probs, 1, generator=rng).item())

    centroids = data[indices].clone()
    for _ in range(n_iters):
        dists = torch.cdist(data, centroids)
        assignments = dists.argmin(dim=1)
        new_centroids = torch.zeros_like(centroids)
        counts = torch.zeros(n_centroids, device=data.device)
        new_centroids.index_add_(0, assignments, data)
        counts.index_add_(0, assignments, torch.ones(N, device=data.device))
        mask = counts > 0
        new_centroids[mask] /= counts[mask].unsqueeze(1)
        new_centroids[~mask] = centroids[~mask]
        centroids = new_centroids
    return centroids

def logit_mse(q, k, k_hat):
    return ((q * (k - k_hat)).sum(dim=1) ** 2).mean().item()

def ssf_log(eigvals, c=5.0):
    shaped = torch.log1p(c * eigvals.clamp(min=0))
    log_shaped = torch.log(shaped.clamp(min=1e-8))
    log_shaped = log_shaped - log_shaped.mean()
    return torch.exp(log_shaped)

def load_model_and_layers(model_name):
    cfg = MODELS[model_name]
    print(f"\n{'='*20} Loading {cfg['id']} {'='*20}")
    try:
        tokenizer = AutoTokenizer.from_pretrained(cfg['id'])
        model = AutoModelForCausalLM.from_pretrained(cfg['id'], torch_dtype=cfg['dtype'], device_map='auto', trust_remote_code=True)
        layers = model.model.layers if hasattr(model, 'model') else model.transformer.h
        full_attn_layers = list(range(len(layers))) if cfg['all_layers_are_full_attn'] else cfg.get('full_attention_layers', [])
        print(f"Detected {len(layers)} layers. Testing full-attention subset: {full_attn_layers[:4]}")
        return model, tokenizer, layers, full_attn_layers
    except Exception as e:
        print(f"Error: {e}"); return None, None, None, None

def extract_traces(model, tokenizer, layers, test_layers):
    traces = {l: {'q': [], 'k': []} for l in test_layers}
    hooks = []
    def get_hook(l, key):
        def hook(m, i, o):
            t = (o[0] if isinstance(o, tuple) else o).detach().cpu().float()
            if key == 'q' and t.shape[-1] > model.config.hidden_size:
                t = t[..., :t.shape[-1] // 2]
            if key == 'k':
                n_q = model.config.num_attention_heads
                n_k = model.config.num_key_value_heads
                if n_q != n_k:
                    t = t.view(t.shape[0], t.shape[1], n_k, -1)
                    t = t.repeat_interleave(n_q // n_k, dim=2)
                    t = t.reshape(t.shape[0], t.shape[1], -1)
            traces[l][key].append(t.reshape(-1, t.shape[-1]))
        return hook

    for l in test_layers:
        hooks.append(layers[l].self_attn.q_proj.register_forward_hook(get_hook(l, 'q')))
        hooks.append(layers[l].self_attn.k_proj.register_forward_hook(get_hook(l, 'k')))

    print(f"Extracting activation traces for {len(test_layers)} layers using WikiText-2...")
    ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
    texts = [x['text'] for x in ds if len(x['text']) > 50][:NUM_SAMPLES]
    inputs = tokenizer(texts, return_tensors='pt', max_length=SEQ_LEN, truncation=True, padding=True).to(device)
    with torch.no_grad(): model(**inputs)
    for h in hooks: h.remove()
    for l in test_layers:
        traces[l]['q'] = torch.cat(traces[l]['q'], 0)
        traces[l]['k'] = torch.cat(traces[l]['k'], 0)
    return traces

def run_benchmark(model_name):
    cfg = MODELS[model_name]; block_dim = cfg['block_dim']
    model, tokenizer, layers, full_attn_layers = load_model_and_layers(model_name)
    if not model: return
    test_layers = full_attn_layers[:4]
    traces = extract_traces(model, tokenizer, layers, test_layers)
    del model; gc.collect(); torch.cuda.empty_cache()

    print("\nStarting Quantization Benchmark...")
    results = []
    for layer in test_layers:
        print(f"  -> Benchmarking Layer {layer}...")
        q, k = traces[layer]['q'], traces[layer]['k']
        N, D = k.shape; n_blocks = D // block_dim; mid = N // 2
        q_cal, k_cal, q_test, k_test = q[:mid], k[:mid], q[mid:], k[mid:]

        # Standard VQ
        k_hat_std = torch.zeros_like(k_test)
        for j in range(n_blocks):
            bj = slice(j*block_dim, (j+1)*block_dim)
            cents = kmeans(k_cal[:, bj], N_CENTROIDS, KMEANS_ITERS, SEED+j)
            k_hat_std[:, bj] = cents[torch.cdist(k_test[:, bj], cents).argmin(1)]
        err_std = logit_mse(q_test, k_test, k_hat_std)

        # SMAQ
        k_test_e, k_hat_e, winvs = torch.zeros_like(k_test), torch.zeros_like(k_test), []
        for j in range(n_blocks):
            bj = slice(j*block_dim, (j+1)*block_dim)
            Sj = (q_cal[:, bj].T @ q_cal[:, bj]) / mid
            ev, evc = torch.linalg.eigh(Sj)
            Wj = evc @ torch.diag(ssf_log(ev).sqrt()) @ evc.T
            winvs.append(evc @ torch.diag(1.0/ssf_log(ev).sqrt()) @ evc.T)
            k_cal_ej = k_cal[:, bj] @ Wj.T
            k_test_e[:, bj] = k_test[:, bj] @ Wj.T
            cents_e = kmeans(k_cal_ej, N_CENTROIDS, KMEANS_ITERS, SEED+j)
            k_hat_e[:, bj] = cents_e[torch.cdist(k_test_e[:, bj], cents_e).argmin(1)]

        k_hat_smaq = torch.zeros_like(k_test)
        for j in range(n_blocks):
            bj = slice(j*block_dim, (j+1)*block_dim)
            k_hat_smaq[:, bj] = k_hat_e[:, bj] @ winvs[j].T

        err_smaq = logit_mse(q_test, k_test, k_hat_smaq)
        gain = (1 - err_smaq/err_std)*100
        results.append([f"L{layer}", f"{err_std:.5f}", f"{err_smaq:.5f}", f"{gain:+.2f}%"])
        print(f"     [DONE] Std: {err_std:.3f}, SMAQ: {err_smaq:.3f}, Gain: {gain:+.2f}%")

    print("\n" + tabulate(results, headers=["Layer", "Std VQ Error", "SMAQ Error", "Gain"]))

def main():
    target = os.environ.get("MODEL", "").strip()
    for m in ([target] if target in MODELS else list(MODELS.keys())): run_benchmark(m)

if __name__ == "__main__":
    main()


==================== Loading Qwen/Qwen3.5-0.8B ====================


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Detected 24 layers. Testing full-attention subset: [3, 7, 11, 15]
Extracting activation traces for 4 layers using WikiText-2...

Starting Quantization Benchmark...
  -> Benchmarking Layer 3...
     [DONE] Std: 1461.717, SMAQ: 484.186, Gain: +66.88%
  -> Benchmarking Layer 7...
     [DONE] Std: 240.493, SMAQ: 109.025, Gain: +54.67%
  -> Benchmarking Layer 11...
     [DONE] Std: 341.184, SMAQ: 88.541, Gain: +74.05%
  -> Benchmarking Layer 15...
     [DONE] Std: 285.266, SMAQ: 74.702, Gain: +73.81%

Layer      Std VQ Error    SMAQ Error  Gain
-------  --------------  ------------  -------
L3             1461.72       484.186   +66.88%
L7              240.493      109.025   +54.67%
L11             341.184       88.5411  +74.05%
L15             285.266       74.7021  +73.81%
